In [0]:
# Step 1: Create own custom schema to improve performance and reduce memory usage
from pyspark.sql.types import *

schema = StructType([
    StructField("Country", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Kilotons of Co2", DoubleType(), True),
    StructField("Metric Tons Per Capita", DoubleType(), True)
])

# Then, read the csv file using the custom schema and convert to a Spark DataFrame

file_location = "/Volumes/karinworkingspace/default/datafortesting/Carbon_(CO2)_Emissions_by_Country.csv"
df = spark.read.csv(file_location, header=True, schema=schema)
display(df.limit(5))

In [0]:
# Step 2: Save the dataframe as a parquet file

df.write.mode("overwrite").format("parquet").save("/Volumes/karinworkingspace/default/datafortesting")



In [0]:
# Step3: Since we changed the format to Parquet, we need to change the file location and read the file using the parquet format
from pyspark.sql.types import *

schema = StructType([
    StructField("Country", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Kilotons of Co2", DoubleType(), True),
    StructField("Metric Tons Per Capita", DoubleType(), True)
])

# Then, read the Parket file using the parquet format

file_location = "/Volumes/karinworkingspace/default/datafortesting/part-00000-tid-742761651420885492-5216e851-8151-43f7-b476-55f6e3da1ae2-126-1.c000.snappy.parquet"
df = spark.read.parquet(file_location)
display(df.limit(5))

In [0]:
# Step4: Implementing GroupBy and aggregation
# ascending=True (default) sorts from smallest to largest, ascending=False sorts from largest to smallest
from pyspark.sql.functions import sum as _sum, col

top10Emitters_Europe_Region = display(
    df.groupBy("Region")
      .agg(_sum("Kilotons of Co2").alias("Total_Co2")).filter(col("Region") == "Europe")
      .orderBy("Total_Co2", ascending=False).limit(10)
)



In [0]:
#Step 5: What are the Top 10 emitters per country?
from pyspark.sql.functions import sum, col

top10Emitters = display(
    df.groupBy("Country")
      .agg(_sum("Kilotons of Co2").alias("Total_Co2"))
      .orderBy("Total_Co2", ascending=False).limit(10)
)

In [0]:
# Step 6: The client is only interested in a specific country ---> Filter the dataframe
df.filter("Country = 'Netherlands'").display()

In [0]:
# Step 7: The client is only interested in The Netherlands for date 01-01-2019.
df.filter("Country = 'Netherlands' AND Date = '01-01-2019'").select("Country", "Date", "Kilotons of Co2").display()

In [0]:
# Final step: Load the Parquet file into a Delta lake table(https://learn.microsoft.com/en-us/azure/databricks/delta/)

# Rename ALL columns with spaces or invalid characters, Delta Lake Tables do not acccept spaces or invalid characters.
df_parquet = df.withColumnRenamed("Kilotons of Co2", "Kilotons_of_Co2") \
    .withColumnRenamed("Metric Tons Per Capita", "Metric_Tons_Per_Capita")

# Since our data is in a Volume and volumes do not support delta transaction tables, convert the data to a managed table and then save as a delta table.
df_parquet.write.format("delta").mode("overwrite").saveAsTable(
    "karinworkingspace.default.datafortesting"
)

# Verify the Delta table
delta_table = spark.read.table("karinworkingspace.default.datafortesting")
display(delta_table)

